<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# ROS 2 Data Paths and Duckiedrone Integration

ROS 2 runs on Linux and provides a software framework for programs to exchange data using shared communication rules. This notebook explains how a robot is organized as a graph of small programs and where common Duckiedrone data enters that graph.

## Why modularity?

A robot may read a camera, estimate its pose, detect objects, choose a behavior, and command motors at the same time. Putting all of that work in one program quickly makes it hard to understand, test, and replace one capability without disturbing the rest.

ROS 2 organizes the same work as a graph of small programs. Each program has one focused responsibility and communicates with the rest of the robot through well-defined interfaces. A camera driver can publish images, a perception node can subscribe to them, and a controller can subscribe to the resulting pose without any one node needing to know how the others are implemented.

This separation lets nodes run in parallel, move to another computer, or be replaced independently. On a Duckiedrone, sensor drivers, perception nodes, and higher-level autonomy nodes can participate in the same ROS 2 graph. The [PX4](https://docs.px4.io/main/) autopilot firmware on the flight controller sends and receives data through [Micro Air Vehicle Link (MAVLink)](https://mavlink.io/) and [MAVROS](https://github.com/mavlink/mavros), which bridges that data into the ROS 2 graph.

## Follow a data path

One possible robot data path appears in [Figure 1](#figure-1).

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    camera driver
      |
      v
    image topic
      |
      v
    perception node
      |
      v
    pose topic
      |
      v
    controller
      |
      v
    command interface
  </pre>
  <figcaption>
    Figure 1: Camera-to-controller data path through the ROS 2 graph.
  </figcaption>
</figure>

Each arrow carries a message with a defined type. The controller needs the pose message's contract, not the perception algorithm's source code. This makes it possible to test or replace one stage while keeping the rest of the graph stable.

## Duckietown Postal Service and ROS 2

Duckietown also uses the [Duckietown Postal Service (DTPS)](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html), a system that lets programs send data to one another. It uses Hypertext Transfer Protocol version 2 (HTTP/2)-compatible messaging, but you do not need to configure it in this learning experience (LX). DTPS and ROS 2 are separate systems: a DTPS path is not automatically a ROS 2 topic.

Two common Duckiedrone data paths appear in [Figure 2](#figure-2).

<figure id="figure-2" class="lx-figure">
  <div style="align-items: flex-start; display: flex; flex-wrap: wrap; gap: 3em; justify-content: center;">
    <pre style="margin:0; text-align:left;">
    camera or time-of-flight (ToF) sensor
      |
      v
    driver
      |
      v
    DTPS
      |
      v
    ROS 2 bridge
      |
      v
    ROS 2 graph
      |
      v
    application node
    </pre>
    <pre style="margin:0; text-align:left;">
    PX4 flight controller
      |
      v
    MAVLink
      |
      v
    MAVROS
      |
      v
    ROS 2 graph
      |
      v
    application node
    </pre>
  </div>
  <figcaption>
    Figure 2: Duckiedrone sensor and flight-controller paths into the ROS 2 graph.
  </figcaption>
</figure>

Camera and ToF drivers publish through DTPS, then `ros2-camera` and `ros2-tof-bottom` republish the data for ROS 2 nodes.

Flight-controller telemetry follows the second path: MAVROS exposes PX4 data, including data from the inertial measurement unit (IMU), directly to ROS 2. An IMU measures acceleration and rotation. This is why later sensor material finds a driver and bridge pair for camera or ToF data, but no separate IMU driver on the Duckiedrone.

### Try it

In the supplied ROS 2 exercise terminal, source the installed distribution and inspect two message types used by the sensor paths above. These commands read local interface definitions; they do not connect to a Duckiedrone or require a running node.

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
ros2 interface show sensor_msgs/msg/CompressedImage
ros2 interface show sensor_msgs/msg/Imu
```

Find the `format` and `data` fields in the compressed image definition and the `orientation`, `angular_velocity`, and `linear_acceleration` fields in the IMU definition. Which side of [Figure 2](#figure-2) corresponds to each type? The definitions show message shape, not whether a sensor is currently publishing.

<details>
<summary>Compare the fields</summary>
<p>The camera path carries compressed image data with a format and byte array; the flight-controller path carries IMU orientation, rotation, and acceleration through MAVROS. The two paths enter ROS 2 through different bridges.</p>
</details>

This LX focuses on ROS 2 because the nodes written here communicate through the ROS 2 graph. The [Duckiedrone IMU Sensors - Learning Experience](https://github.com/duckietown/lx-dd-sensors-imu) and [Duckiedrone ToF Sensors - Learning Experience](https://github.com/duckietown/lx-dd-sensors-tof) identify the sensor-specific messages and configuration on each path.

## Further reading

The official ROS 2 Jazzy documentation on [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html) explains the streaming interface used in these data paths.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
